# Data Pipeline — EcoTravel Agent

This notebook demonstrates the data pipeline that feeds the EcoTravel Agent.
Four external APIs and one local dataset are combined to produce crowd-scored hotel recommendations.

## Data Sources
1. **TripAdvisor Content API** — Hotel search, ratings, and reviews
2. **Amadeus Hotel Search API** — Room availability and pricing (sandbox)
3. **OpenWeatherMap API** — Weather forecast and suitability scoring
4. **PredictHQ Events API** — Local events magnitude scoring
5. **GeoNames Dataset** — Nearby city lookup (cities500.txt, download separately)

## How to run
- Copy `.env.example` to `.env` and fill in your API keys
- Download GeoNames data: `cities500.zip` from geonames.org/export/dump/ → unzip to `data/`
- Run all cells top to bottom

In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path so src imports work from the notebook
sys.path.insert(0, str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / ".env")

from src.clients.tripadvisor import TripAdvisorClient
from src.clients.amadeus import AmadeusClient
from src.clients.weather import WeatherClient
from src.clients.events import EventsClient
from src.clients.geonames import GeoNamesClient
from src.scoring.crowd_scorer import calculate_crowd_score

# Initialize clients — API keys loaded from .env
ta = TripAdvisorClient(os.environ["TRIPADVISOR_API_KEY"])
amadeus = AmadeusClient(os.environ["AMADEUS_CLIENT_ID"], os.environ["AMADEUS_CLIENT_SECRET"])
weather = WeatherClient(os.environ["OPENWEATHERMAP_API_KEY"])
events = EventsClient(os.environ["PREDICTHQ_API_KEY"])
geo = GeoNamesClient(Path.cwd().parent / "data" / "cities500.txt")

print("All clients initialized successfully.")

## Source 1: TripAdvisor — Hotel Search

TripAdvisor Content API v2 searches for hotels within a radius of the target city.
Returns hotel names, location IDs, and ratings for use in the agent.

In [ ]:
LOCATION = "Asheville, NC"
RADIUS = 25

ta_results = ta.search_hotels(location=LOCATION, radius_miles=RADIUS)
print(f"Found {len(ta_results)} hotels near {LOCATION} within {RADIUS} miles.")

if ta_results:
    ta_df = pd.DataFrame([
        {
            "location_id": h.get("location_id", ""),
            "name": h.get("name", ""),
            "rating": h.get("rating", ""),
        }
        for h in ta_results[:5]
    ])
    display(ta_df)
else:
    print("No results (check API key or try a different location)")

## Source 2: Amadeus — Hotel Availability & Pricing

Amadeus Hotel Search API (sandbox) returns available room offers with pricing.
`estimate_availability_pct()` converts offer count to an estimated availability percentage.

In [ ]:
CHECKIN = "2026-08-01"
CHECKOUT = "2026-08-05"

amadeus_results = amadeus.search_hotel_offers(
    city_code="AVL", checkin_date=CHECKIN, checkout_date=CHECKOUT
)
print(f"Found {len(amadeus_results)} hotel offers from Amadeus (sandbox).")

for offer in amadeus_results[:3]:
    avail = amadeus.estimate_availability_pct(offer)
    rate = offer["offers"][0]["price"]["total"] if offer.get("offers") else "N/A"
    print(f"  {offer['hotel'].get('name', 'Unknown')}: {avail:.0f}% available, ${rate}/night")

if not amadeus_results:
    print("Sandbox may return empty results — this is normal in the test environment.")

## Source 3: OpenWeatherMap — Weather Suitability

Returns a 0.0–1.0 suitability score based on the proportion of good vs. bad forecast conditions.
Good = Clear/Clouds. Bad = Rain/Snow/Thunderstorm/Drizzle/Tornado/Squall.

In [ ]:
# Asheville, NC coordinates
LAT, LON = 35.5951, -82.5515

weather_score = weather.get_weather_suitability(LAT, LON, CHECKIN, CHECKOUT)
print(f"Weather suitability score for {LOCATION} ({CHECKIN} to {CHECKOUT}): {weather_score:.2f}")
print("  0.0 = severe weather, 0.5 = neutral/no data, 1.0 = ideal conditions")

## Source 4: PredictHQ — Local Events

Events API returns magnitude (0–1) representing crowd impact from local events.
Events with rank ≥ 60 are listed as significant crowd drivers.

In [ ]:
event_score, event_list = events.get_event_magnitude(LOCATION, CHECKIN, CHECKOUT)
print(f"Event magnitude score: {event_score:.2f}")
if event_list:
    print("Significant events that may drive crowds:")
    for evt in event_list:
        print(f"  - {evt}")
else:
    print("No major events detected for these dates.")

## Source 5: GeoNames — Nearby Low-Crowd Destinations

GeoNames cities500 dataset provides cities within a radius, classified by population size.
Population < 10,000 = low crowd indicator. 10,000–100,000 = medium. 100,000+ = high.

In [ ]:
nearby = geo.find_nearby_destinations(center_lat=LAT, center_lng=LON, radius_miles=50, limit=8)

if nearby:
    nearby_df = pd.DataFrame([d.model_dump() for d in nearby])
    print(f"Found {len(nearby)} nearby destinations within 50 miles:")
    display(nearby_df[["name", "distance_miles", "population", "crowd_indicator"]])
else:
    print("GeoNames data not available. Download cities500.txt from geonames.org and place in data/")

## Combined Pipeline: Crowd-Scored Hotel Results

This cell ties all sources together to produce the final output: a crowd-scored hotel table.
Only hotels with >20% availability are shown (the 80% booking filter).
Results are sorted by crowd score ascending (lower = less crowded).

In [ ]:
rows = []

for hotel in ta_results[:8]:
    avail_pct = 45.0   # Default when Amadeus sandbox returns no data
    avg_rate = 150.0

    if amadeus_results:
        offer = amadeus_results[0]
        avail_pct = amadeus.estimate_availability_pct(offer)
        if offer.get("offers"):
            avg_rate = float(offer["offers"][0]["price"]["total"])

    if avail_pct < 20.0:   # Enforce the >20% availability filter
        continue

    score = calculate_crowd_score(
        availability_pct=avail_pct,
        avg_nightly_rate=avg_rate,
        baseline_rate=avg_rate * 0.85,
        event_magnitude=event_score,
        seasonal_demand=0.6,        # Summer = moderate peak season
        weather_suitability=weather_score,
    )

    rows.append({
        "Hotel": hotel.get("name", "Unknown"),
        "Availability %": f"{avail_pct:.0f}%",
        "% Booked": f"{100 - avail_pct:.0f}%",
        "Avg Rate/Night": f"${avg_rate:.0f}",
        "Rating": hotel.get("rating", "N/A"),
        "Crowd Score": score.total,
        "Crowd Factors": "; ".join(score.crowd_reasons) if score.crowd_reasons else "None",
    })

if rows:
    results_df = pd.DataFrame(rows).sort_values("Crowd Score")
    print(f"Showing {len(results_df)} hotels with >20% availability, sorted by crowd score:")
    display(results_df)
else:
    print("No results to display. Check API keys and try running individual cells above.")